<!-- MIE690A article-aligned validation v3 -->

# Week 9 Lab 3 — Audit a moving-throat nozzle dataset

<!-- FLOWMLLAB_COLAB_LAUNCH_V1 -->
[Open in Colab](https://colab.research.google.com/github/Ehsan-Roohi/FlowMLLab/blob/main/notebooks/week09/W9_Lab3_Nozzle_Data_Alignment_Audit.ipynb)

This short CPU exercise extends the Week 4 data contract. It deliberately introduces and catches a coordinate bug before any neural training. No student code, correspondence, or results are included.

**Objectives:** distinguish fixed branch sensors from case-dependent queries; identify a wrong pairing despite correct shapes; repair the pairings; verify the isentropic reference. This is an ideal-gas, inviscid, choked, shock-free quasi-1D benchmark, not a DSMC or multidimensional CFD validation and not an evaluation of a trained surrogate.

For case c and point j, the pair is `(branch[c], x[c,j]) -> q[c,j]`. Each branch contains 41 geometry values sampled at the SAME sensor positions plus p0 and T0. Each query grid includes its own throat exactly. A low training loss cannot certify this pairing.

**Prediction question:** all arrays have the expected shapes. What happens if every nozzle is assigned the first nozzle's x coordinates? Explain before running the negative control.


In [1]:
# FLOWMLLAB_COLAB_BOOTSTRAP_V1
from pathlib import Path
import sys, subprocess, os
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    root = Path("/content/FlowMLLab")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Ehsan-Roohi/FlowMLLab.git", str(root)], check=True)
    os.chdir(root)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common/deeponet_data_audit.py").exists())
sys.path.insert(0, str(ROOT / "common"))
import numpy as np
from scipy.optimize import brentq
from deeponet_data_audit import audit_pairs, audit_fixed_sensors, audit_case_splits


In [2]:
GAMMA, GAS_R = 1.4, 287.05
sensors = np.linspace(0., 1., 41)

def area(x, throat, exit_ratio):
    x = np.asarray(x)
    return np.where(x <= throat,
        1 + .6*(1 + np.cos(np.pi*x/throat)),
        1 + .5*(exit_ratio-1)*(1-np.cos(np.pi*(x-throat)/(1-throat))))

def area_mach(m):
    return (2/(GAMMA+1)*(1+(GAMMA-1)*m*m/2))**((GAMMA+1)/(2*(GAMMA-1)))/m

def reference(throat, exit_ratio, p0, T0):
    # 41 points in total, including the exact throat for every case.
    n_left = round(40*throat)+1
    x = np.r_[np.linspace(0, throat, n_left), np.linspace(throat, 1, 42-n_left)[1:]]
    A = area(x, throat, exit_ratio)
    M = np.array([1. if abs(xj-throat)<1e-12 else brentq(
        lambda m: area_mach(m)-Aj, *( (1e-7, 1.) if xj<throat else (1., 10.) ))
        for xj, Aj in zip(x, A)])
    theta = 1+(GAMMA-1)*M*M/2
    T = T0/theta
    p = p0/theta**(GAMMA/(GAMMA-1))
    rho = p/(GAS_R*T)
    u = M*np.sqrt(GAMMA*GAS_R*T)
    return x, A, np.column_stack([M, p, T, rho, u])

parameters = [(.35, 2., 400000., 300.), (.40, 2.5, 500000., 320.), (.46, 3.5, 700000., 400.)]
records = [reference(*params) for params in parameters]
branch = np.array([np.r_[area(sensors, t, ae), p0, T0] for t, ae, p0, T0 in parameters])
source_x = np.concatenate([r[0] for r in records])  # kept from source records
source_ids = np.repeat(["nozzle-a", "nozzle-b", "nozzle-c"], 41)
target = np.concatenate([r[2] for r in records])
branch_rows = np.repeat(branch, 41, axis=0)
branch_ids = np.repeat(["nozzle-a", "nozzle-b", "nozzle-c"], 41)
assert branch_rows.shape == (123, 43) and target.shape == (123, 5)
audit_fixed_sensors(np.tile(sensors, (3, 1)))


In [3]:
# Intentional bug: correct shape, wrong physical coordinate correspondence.
wrong_x = np.tile(records[0][0], len(records))
assert wrong_x.shape == source_x.shape
try:
    audit_pairs(branch_ids, source_ids, wrong_x, source_x)
except ValueError as error:
    print("Expected failure:", error)
else:
    raise AssertionError("Intentional coordinate bug was not detected")
print("Maximum coordinate displacement:", float(np.max(np.abs(wrong_x-source_x))))

# A second, independent error: variable query locations used as branch sensors.
try:
    audit_fixed_sensors(np.stack([r[0] for r in records]))
except ValueError as error:
    print("Expected sensor failure:", error)
else:
    raise AssertionError("Variable sensor locations were not detected")


Expected failure: Query coordinates do not match the source target coordinates
Maximum coordinate displacement: 0.010000000000000064
Expected sensor failure: Branch sensor locations/order must be fixed across cases


In [4]:
# Repair: retain each target's own query location; branch sensors stay fixed.
correct_x = np.concatenate([record[0].copy() for record in records])
print("Repaired pairing:", audit_pairs(branch_ids, source_ids, correct_x, source_x))
for params, (x, A, q) in zip(parameters, records):
    M, p, T, rho, u = q.T
    flux = rho*u*A  # mass flow per unit throat area, since A is A/At
    cv = flux.std()/flux.mean()
    throat_error = abs(M[np.argmin(abs(x-params[0]))]-1)
    residual = np.max(np.abs(area_mach(M)-A))
    assert cv < 1e-9 and throat_error < 1e-12 and residual < 1e-9
    assert np.all(q[:, 1:4] > 0)
    print(f"throat={params[0]:.2f}: mass-flow CV={cv:.3e}, sonic error={throat_error:.1e}, area-Mach residual={residual:.3e}")


Repaired pairing: {'rows': 123, 'max_coordinate_error': 0.0}
throat=0.35: mass-flow CV=1.345e-13, sonic error=0.0e+00, area-Mach residual=1.301e-12
throat=0.40: mass-flow CV=1.758e-13, sonic error=0.0e+00, area-Mach residual=1.316e-12
throat=0.46: mass-flow CV=1.197e-13, sonic error=0.0e+00, area-Mach residual=1.239e-12


## Interpret and extend

1. The zero coordinate error after repair certifies correspondence only; it is not neural accuracy. The physics checks above apply to the reference, not a surrogate.
2. Alternative repair: evaluate reference fields directly on a common fixed query grid. Since an area–Mach reference is available, direct evaluation avoids interpolation error. If interpolating CFD instead, quantify interpolation error separately. Branch sensors and queries need not coincide.
3. `np.repeat(branch, 41, axis=0)` repeats each case for its points. `np.tile(branch, (41,1))` interleaves cases. Both produce `(123,43)`; check case keys to distinguish them.
4. Train/validation/test split by globally unique physical case keys before point expansion. Seed and case-number prefixes alone cannot establish that physical parameters are different. Fit normalization and POD bases only within the allowed fit split.
5. For the optional ML extension, compare coordinate MLP and DeepONet on identical splits and targets. Report per-case relative L2 mean/std/max, positivity, mass-flow variation, sonic error, and warmed-up full-profile inference time. Use validation for early stopping. Freeze both models before evaluating fresh test cases; previously inspected test cases are diagnostic data.
6. A fixed-sensor representation of a two-parameter nozzle family supports only that family. Test different shape families separately before claiming general geometry learning. Five outputs can share `b*t` with five linear heads; this is not five independent scalar dot products accidentally collapsed to one output.

**Deliverable:** identify the two intentionally failing contracts, explain the repair, and distinguish reference physics checks from surrogate validation. No claim that DeepONet outperforms MLP is made in this exercise.
